In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt 
import matplotlib.patches as mpatches 
import csv 
import numpy as np 
import seaborn as sns 
from sklearn.metrics import root_mean_squared_error
import ast 


In [ ]:
TASC_VColor         = "#57B8FF"  
TASC_permute_VColor = "#2176AE" 
SC_VColor           = "#7ED957" 
RSC_VColor          = "#FF9F40"  
CIM_VColor          = "#E255A1"  

XLABEL_FS = 14
YLABEL_FS = 14
YTICKS_FS = 14
XTICKS_FS = 14
LEGEND_FS = 14

In [ ]:
def process_tasc_df(df):
    list_cols = ["trueTarget", "pred_tasc"]
    for col in list_cols:
        df[col] = df[col].apply(ast.literal_eval)

    df["rmse_pred_tasc"] = df.apply(
        lambda row: root_mean_squared_error(row["trueTarget"][row["T0"]:], row["pred_tasc"][row["T0"]:]),
        axis=1
    )

    return pd.DataFrame({
        "RMSE": df["rmse_pred_tasc"],
        "Method": ["TASC"] * len(df),
        "n": df["n"].tolist(),
        "T0": df["T0"].tolist(),
        "d": df["d"].tolist()
    })

def process_cim_df(df):
    list_cols = ["trueTarget", "pred_cim"]
    for col in list_cols:
        df[col] = df[col].apply(ast.literal_eval)

    df["rmse_pred_cim"] = df.apply(
        lambda row: root_mean_squared_error(row["trueTarget"][row["T0"]:], row["pred_cim"][row["T0"]:]),
        axis=1
    )
    
    return pd.DataFrame({
        "RMSE": df["rmse_pred_cim"],
        "Method": ["CIM"] * len(df), 
        "n": df["n"].tolist(),
        "T0": df["T0"].tolist(),
        "d": df["d"].tolist()
    })

def process_benchmark_df(df):
    list_cols = ["trueTarget", "pred_rsc", "pred_sc"]
    for col in list_cols:
        df[col] = df[col].apply(ast.literal_eval)

    df["rmse_pred_sc"] = df.apply(
        lambda row: root_mean_squared_error(row["trueTarget"][row["T0"]:], row["pred_sc"][row["T0"]:]),
        axis=1
    )
    df["rmse_pred_rsc"] = df.apply(
        lambda row: root_mean_squared_error(row["trueTarget"][row["T0"]:], row["pred_rsc"][row["T0"]:]),
        axis=1
    )

    return pd.DataFrame({
        "RMSE": np.concatenate([df["rmse_pred_sc"], df["rmse_pred_rsc"]]),
        "Method": (
            ["SC"] * len(df["rmse_pred_sc"]) +
            ["RSC"] * len(df["rmse_pred_rsc"])
        ),
        "n": df["n"].tolist() * 2,
        "T0": df["T0"].tolist() * 2,
        "d": df["d"].tolist() * 2
    })

files = [
    "resultLogscricket/cricket_placebo_test_tasc_n18_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_tasc_n36_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_tasc_n72_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_tasc_n144_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_tasc_n288_d5_T072_aistats.csv"
]

optimal_benchmark_files_rsc_sc = [
    "resultLogscricket/cricket_placebo_test_rsc_sc_n18_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_rsc_sc_n36_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_rsc_sc_n72_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_rsc_sc_n144_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_rsc_sc_n288_d5_T072_aistats.csv"
]

optimal_benchmark_files_cim = [
    "resultLogscricket/cricket_placebo_test_cim_n18_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_cim_n36_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_cim_n72_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_cim_n144_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_cim_n288_d5_T072_aistats.csv"
]

tasc_dfs = [process_tasc_df(pd.read_csv(f)) for f in files]
benchmark_dfs = [process_benchmark_df(pd.read_csv(f)) for f in optimal_benchmark_files_rsc_sc]
cim_dfs = [process_cim_df(pd.read_csv(f)) for f in optimal_benchmark_files_cim]

df_all = pd.concat(tasc_dfs + benchmark_dfs + cim_dfs, ignore_index=True)

T0 = df_all["T0"].unique().tolist()[0]
d = df_all["d"].unique().tolist()[0]

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_all,
    x="n",           
    y="RMSE",
    hue="Method",
    whis=[0, 95],
    palette={
        "TASC": TASC_VColor,
        "SC": SC_VColor,
        "RSC": RSC_VColor,
        "CIM": CIM_VColor
    },
)

plt.xlabel("Number of Donors (n)", fontsize=XLABEL_FS)
plt.ylabel("RMSE", fontsize=YLABEL_FS)
plt.ylim(0, 40)
plt.yticks(fontsize=YTICKS_FS)
plt.xticks(fontsize=XTICKS_FS)
plt.legend(title="Method", fontsize=LEGEND_FS, loc='upper left', title_fontsize=LEGEND_FS)
plt.tight_layout()
plt.minorticks_on()
plt.tick_params(axis="x", which="minor", bottom=False, top=False)   
plt.tick_params(axis="y", which="minor", left=True, right=False)    

plt.grid(axis='y', which="major", linestyle='-', linewidth=0.8, alpha=0.7)
plt.grid(axis='y', which="minor", linestyle='--', linewidth=0.5, alpha=0.4)

plt.show()
medians = df_all.groupby(["n", "Method"])["RMSE"].median().reset_index()
print(medians)

In [ ]:
def plot_horizon_plot_from_df_bench(df_tasc, df_bench, df_cim, save_path=None):
    list_cols = ["trueTarget", "pred_sc", "pred_rsc"]
    for col in list_cols:
        df_bench[col] = df_bench[col].apply(ast.literal_eval)

    list_cols_tasc = ["trueTarget", "pred_tasc"]
    for col in list_cols_tasc:
        df_tasc[col] = df_tasc[col].apply(ast.literal_eval)

    list_cols_cim = ["trueTarget", "pred_cim"]
    for col in list_cols_cim:
        df_cim[col] = df_cim[col].apply(ast.literal_eval)

    block_size = 6   
    T = 120        
    rmse_data = []

    for _, row in df_tasc.iterrows():
        T0 = int(row["T0"])
        d = int(row["d"])
        n = int(row["n"])
        true_target = np.array(row["trueTarget"])
        tasc_pred = np.array(row["pred_tasc"])
        for start in range(T0, T, block_size):
            end = min(start + block_size, T)

            rmse_data.append({
                "RMSE": root_mean_squared_error(true_target[start:end], tasc_pred[start:end]),
                "Method": "TASC",
                "Block": (start - T0) // block_size + 1,
                "n": n
            })

    for _, row in df_bench.iterrows():
        T0 = int(row["T0"])
        d = int(row["d"])
        n = int(row["n"])
        true_target = np.array(row["trueTarget"])
        sc_pred = np.array(row["pred_sc"])
        rsc_pred = np.array(row["pred_rsc"])

        for start in range(T0, T, block_size):
            end = min(start + block_size, T)

            rmse_data.append({
                "RMSE": root_mean_squared_error(true_target[start:end], sc_pred[start:end]),
                "Method": "SC",
                "Block": (start - T0) // block_size + 1,
                "n": n
            })

            rmse_data.append({
                "RMSE": root_mean_squared_error(true_target[start:end], rsc_pred[start:end]),
                "Method": "RSC",
                "Block": (start - T0) // block_size + 1,
                "n": n
            })

    for _, row in df_cim.iterrows():
        T0 = int(row["T0"])
        d = int(row["d"])
        n = int(row["n"])
        true_target = np.array(row["trueTarget"])
        cim_pred = np.array(row["pred_cim"])
        for start in range(T0, T, block_size):
            end = min(start + block_size, T)

            rmse_data.append({
                "RMSE": root_mean_squared_error(true_target[start:end], cim_pred[start:end]),
                "Method": "CIM",
                "Block": (start - T0) // block_size + 1,
                "n": n
            })
    rmse_df = pd.DataFrame(rmse_data)

    plt.figure(figsize=(12, 6))
    ax = sns.boxplot(
        data=rmse_df,
        x="Block",
        y="RMSE",
        hue="Method",
        whis=[0, 95],
        palette={"TASC": TASC_VColor, "SC": SC_VColor, "RSC": RSC_VColor, "CIM": CIM_VColor},
    )

    num_blocks = len(rmse_df["Block"].unique())
    xtick_labels = [f"{1 + i*block_size}-{(i+1)*block_size}" for i in range(num_blocks)]
    xtick_labels = ["13", "14", "15", "16", "17", "18", "19", "20"]
    ax.set_xticks(range(num_blocks))
    ax.set_xticklabels(xtick_labels, fontsize=14)

    plt.xlabel("Prediction Horizon (over)", fontsize=14)
    plt.ylabel("RMSE", fontsize=14)
    plt.yticks(fontsize=14)
    plt.ylim(0, 40)
    plt.legend(title="Method", fontsize=12, loc='upper left', title_fontsize=14)

    plt.minorticks_on()
    plt.tick_params(axis="x", which="minor", bottom=False, top=False)   
    plt.tick_params(axis="y", which="minor", left=True, right=False)     

    # Major + minor gridlines only for y-axis
    plt.grid(axis='y', which="major", linestyle='-', linewidth=0.8, alpha=0.7)
    plt.grid(axis='y', which="minor", linestyle='--', linewidth=0.5, alpha=0.4)

    plt.tight_layout()

    # Save or show
    if save_path:
        plt.savefig(save_path)
        print(f"Saved horizon plot to {save_path}")
    plt.show()


df_tasc = pd.read_csv("resultLogscricket/cricket_placebo_test_tasc_n144_d5_T072_aistats.csv")
df_bench = pd.read_csv("resultLogscricket/cricket_placebo_test_rsc_sc_n144_d5_T072_aistats.csv")
df_cim = pd.read_csv("resultLogscricket/cricket_placebo_test_cim_n144_d5_T072_aistats.csv")

plot_horizon_plot_from_df_bench(df_tasc, df_bench, df_cim, save_path=None)

In [ ]:
def process_tasc_df(df):
    list_cols = ["trueTarget", "pred_tasc", "tasc_target_var_estimates", "R_tasc"]
    for col in list_cols:
        df[col] = df[col].apply(ast.literal_eval)
    
    df["tasc_upper"] = df.apply(lambda row: row["pred_tasc"] + 1.96 * np.sqrt(row["tasc_target_var_estimates"]), axis=1)
    df["tasc_lower"] = df.apply(lambda row: row["pred_tasc"] - 1.96 * np.sqrt(row["tasc_target_var_estimates"]), axis=1)
    df["tasc_upper"] = df["tasc_upper"].apply(np.array)
    df["tasc_lower"] = df["tasc_lower"].apply(np.array)
    df["ci_width"] = df.apply(lambda row: row["tasc_upper"] - row["tasc_lower"], axis=1)
    df["ci_post"] = df["ci_width"].apply(lambda x: np.mean(x[72: ])) #because this is intervention point = 72

    df["rmse_pred_tasc"] = df.apply(
        lambda row: root_mean_squared_error(row["trueTarget"][row["T0"]:], row["pred_tasc"][row["T0"]:]),
        axis=1
    )

    return pd.DataFrame({
        "RMSE": df["rmse_pred_tasc"],
        "Method": ["TASC"] * len(df),
        "n": df["n"].tolist(),
        "T0": df["T0"].tolist(),
        "d": df["d"].tolist(),
        "ci": df["ci_post"].tolist()
    })

def process_benchmark_df(df):
    list_cols = ["trueTarget", "pred_cim", "cim_posterior_lower", "cim_posterior_upper"]
    for col in list_cols:
        df[col] = df[col].apply(ast.literal_eval)

    df["rmse_pred_cim"] = df.apply(
        lambda row: root_mean_squared_error(row["trueTarget"][row["T0"]:], row["pred_cim"][row["T0"]:]),
        axis=1
    )
    df["cim_lower"] = df["cim_posterior_lower"].apply(np.array)
    df["cim_upper"] = df["cim_posterior_upper"].apply(np.array)
    df["ci_width"] = df.apply(lambda row: row["cim_upper"] - row["cim_lower"], axis=1) 

    df["ci_post"] = df["ci_width"].apply(lambda x: np.mean(x[72: ])) #because this is intervention point = 72

    return pd.DataFrame({
        "RMSE": df["rmse_pred_cim"],
        "Method": ["CIM"] * len(df["rmse_pred_cim"]),
        "n": df["n"].tolist(),
        "T0": df["T0"].tolist(),
        "d": df["d"].tolist(),
        "ci": df["ci_post"].tolist()
    })

files = [
    "resultLogscricket/cricket_placebo_test_tasc_n18_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_tasc_n36_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_tasc_n72_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_tasc_n144_d5_T072_aistats.csv"
]

optimal_benchmark_files = [
    "resultLogscricket/cricket_placebo_test_cim_n18_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_cim_n36_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_cim_n72_d5_T072_aistats.csv",
    "resultLogscricket/cricket_placebo_test_cim_n144_d5_T072_aistats.csv"
]

tasc_dfs = [process_tasc_df(pd.read_csv(f)) for f in files]
benchmark_dfs = [process_benchmark_df(pd.read_csv(f)) for f in optimal_benchmark_files]

df_all = pd.concat(tasc_dfs + benchmark_dfs, ignore_index=True)

T0 = df_all["T0"].unique().tolist()[0]
d = df_all["d"].unique().tolist()[0]

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_all,
    x="n",            # Number of donors on x-axis
    y="ci",
    hue="Method",
    whis=[0, 95],
    palette={
        "TASC": TASC_VColor,
        "CIM": CIM_VColor,
    },
)

plt.xlabel("# Donors", fontsize=XLABEL_FS+4)
plt.ylabel("Confidence Interval Width", fontsize=YLABEL_FS+4)
plt.ylim(0, 35)
plt.yticks(fontsize=YTICKS_FS+4)
plt.xticks(fontsize=XTICKS_FS+4)
plt.legend(title="Method", fontsize=LEGEND_FS+4, loc='upper left', title_fontsize=LEGEND_FS+4)
plt.tight_layout()
plt.minorticks_on()
plt.grid(axis='y', which="major", linestyle='-', linewidth=0.8, alpha=0.7)
plt.grid(axis='y', which="minor", linestyle='--', linewidth=0.5, alpha=0.4)
plt.show()

In [ ]:
df = pd.read_csv("resultLogscricket/cricket_placebo_coeff_rsc_n72_d5_T058_aistats.csv")

list_cols = ["trueTarget", "pred_rsc_0_1", "pred_rsc_1_0", "pred_rsc_10_0", "pred_rsc_100_0", "pred_rsc_1000_0", "pred_rsc_10000_0", "pred_rsc_100000_0", "pred_rsc_1000000_0"]

for col in list_cols:
    df[col] = df[col].apply(ast.literal_eval)

pred_cols = ['pred_rsc_0_1', 'pred_rsc_1_0', 'pred_rsc_10_0', 'pred_rsc_100_0', 'pred_rsc_1000_0', 'pred_rsc_10000_0', 'pred_rsc_100000_0', 'pred_rsc_1000000_0']
for col in pred_cols: 
    df["rmse_"+col] = df.apply(
        lambda row: root_mean_squared_error(
            row["trueTarget"][row["T0"]:], row[col][row["T0"]:]
        ),
        axis=1
    )


df = pd.DataFrame({
    "RMSE": np.concatenate([df["rmse_pred_rsc_0_1"], df["rmse_pred_rsc_1_0"], df["rmse_pred_rsc_10_0"], df["rmse_pred_rsc_100_0"], df["rmse_pred_rsc_1000_0"], df["rmse_pred_rsc_10000_0"], df["rmse_pred_rsc_100000_0"], df["rmse_pred_rsc_1000000_0"]]),
    "Method": ["RIDGE-0.1"] * len(df["rmse_pred_rsc_0_1"]) + ["RIDGE-1.0"] * len(df["rmse_pred_rsc_1_0"]) +
                 ["RIDGE-10.0"] * len(df["rmse_pred_rsc_10_0"]) +
                 ["RIDGE-100.0"] * len(df["rmse_pred_rsc_100_0"]) +  
                 ["RIDGE-1000.0"] * len(df["rmse_pred_rsc_1000_0"]) +  
                 ["RIDGE-10000.0"] * len(df["rmse_pred_rsc_10000_0"]) +  
                 ["RIDGE-100000.0"] * len(df["rmse_pred_rsc_100000_0"]) +
                 ["RIDGE-1000000.0"] * len(df["rmse_pred_rsc_1000000_0"])  
})

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df,
    x="Method", 
    y="RMSE",
    hue="Method",
    whis=[0, 95],
    palette={"RIDGE-0.1": "#1f77b4", "RIDGE-1.0": "#ff7f0e", "RIDGE-10.0": "#ff0eb3", "RIDGE-100.0": "#8BAE21", "RIDGE-1000.0": "#2FF02C", "RIDGE-10000.0": "#8899AA", "RIDGE-100000.0": "#668888", "RIDGE-1000000.0": "#229988"},
    legend=False,
)

plt.xlabel("Method-Lambda")
plt.ylim(bottom=0)                
plt.tight_layout()
plt.show()
medians = df.groupby(["Method"])["RMSE"].median().reset_index()
print(medians)
maximums = df.groupby(["Method"])["RMSE"].max().reset_index()
print(maximums)